# Evidence snapshot to Drive (YL-004B, stage 02)

For one main company: takes its sources from the database, makes a dated text snapshot of each page (no AI) and saves **one file `<company_id>.md`** in the Drive folder **YL004B evidence**.
The Make Teaser Generator reads this file from that folder. Nothing is copied by hand.

**Before the first run (once):**
1. Colab Secrets (key icon on the left): add `SUPABASE_DB_URL` = the connection string of the Supabase project "Yl004B prototype" (Connect → Session pooler, with the password). Switch on "Notebook access". Never paste it into a cell.
2. The Google account you sign in with in Step 2 must be able to edit the folder YL004B evidence.

Run the cells from top to bottom.

### Step 1. Get the code

In [ ]:
import os

REPO_URL = "https://github.com/Yul04ek/ai-content-claim-check.git"
BRANCH = "main"
FOLDER = "/content/claim-check"

if not os.path.exists(FOLDER):
    !git clone -q -b {BRANCH} {REPO_URL} {FOLDER}
else:
    !git -C {FOLDER} pull -q origin {BRANCH}
%cd {FOLDER}
!git log --oneline -1
!pip install -q requests beautifulsoup4 lxml psycopg2-binary

### Step 2. Sign in to Google (Drive access)
A window asks which Google account to use. Choose the account that can edit the folder YL004B evidence.

In [ ]:
from google.colab import auth
from googleapiclient.discovery import build

auth.authenticate_user()
drive = build("drive", "v3")

EVIDENCE_FOLDER_ID = "14vBlGO9eX8v5N8MMGNHpEE1BtmK7glVi"   # folder "YL004B evidence"

# Check that the folder is visible to this account
folder = drive.files().get(fileId=EVIDENCE_FOLDER_ID, fields="name, capabilities(canAddChildren)",
                           supportsAllDrives=True).execute()
print("Folder:", folder["name"], "| can write:", folder["capabilities"]["canAddChildren"])

### Step 3. Choose the company
Run once with `COMPANY_ID = ""`: the cell lists the companies that are ready for a teaser (reviewed, no teaser yet).
Copy one `company_id` into `COMPANY_ID` and run the cell again.

In [ ]:
import psycopg2
from google.colab import userdata

COMPANY_ID = ""   # e.g. "G-5473514d-876d-4d39-badc-636e661bd8db"

def db_rows(sql, params=()):
    """Run one read-only query and return the rows as dicts."""
    with psycopg2.connect(userdata.get("SUPABASE_DB_URL")) as conn:
        conn.set_session(readonly=True)          # this notebook never changes the database
        with conn.cursor() as cur:
            cur.execute(sql, params)
            cols = [c.name for c in cur.description]
            return [dict(zip(cols, r)) for r in cur.fetchall()]

candidates = db_rows("select company_id, label, review_state, sources_for_teaser "
                     "from public.teaser_candidates order by label")
print("Ready for a teaser:")
for c in candidates:
    print(f"  {c['company_id']}  {c['label']}  ({c['sources_for_teaser']} sources, {c['review_state']})")

if COMPANY_ID:
    sources = db_rows("select evidence_id, belongs_to, label_in_source, source_url, source_type "
                      "from public.teaser_sources where company_id = %s order by evidence_id", (COMPANY_ID,))
    label = db_rows("select label from public.companies where id = %s", (COMPANY_ID,))
    assert label, f"No company with id {COMPANY_ID}"
    LABEL = label[0]["label"]
    if COMPANY_ID not in {c["company_id"] for c in candidates}:
        print(f"\nNOTE: {LABEL} is not in the list above (not reviewed, a duplicate, or it already has a teaser).")
    print(f"\n{LABEL}: {len(sources)} sources (all companies of the group, identical addresses counted once)")
    for s in sources:
        own = "" if s["belongs_to"] == COMPANY_ID else f"   [from duplicate {s['label_in_source']}]"
        print(f"  {s['evidence_id']}  {s['source_url']}{own}")

### Step 4. Read the pages
Every address is read once. Pages that cannot be read are listed as SKIPPED with the reason (not https, forbidden by robots.txt, page not found, built by scripts).

In [ ]:
import sys
sys.path.insert(0, "/content/claim-check")
from evidence_fetch.fetch import fetch_snapshot

assert COMPANY_ID, "Set COMPANY_ID in Step 3 first."
snaps, skipped = {}, []          # snaps: evidence_id -> snapshot
for s in sources:
    url = s["source_url"].strip()
    try:
        snap = fetch_snapshot(url)
    except Exception as e:
        skipped.append(s)
        print(f"SKIPPED {s['evidence_id']} {url}\n   reason: {e}")
        continue
    words = len(snap["text"].split())
    if words < 80:               # pages built by scripts give almost no text
        skipped.append(s)
        print(f"SKIPPED {s['evidence_id']} {url}\n   reason: only {words} words, the page is probably built by scripts")
        continue
    snaps[s["evidence_id"]] = snap
    print(f"OK      {s['evidence_id']} {url}: {words} words, page date: {snap['page_date'] or 'none shown'}")
print(f"\n{len(snaps)} read, {len(skipped)} skipped")

### Step 5 (only if pages were skipped). Saved pages instead
For each skipped address: open it in the browser, save it (Ctrl+S, "Webpage, HTML only"), then upload the file when the cell asks.
To leave an address out, press Cancel in the upload window.

In [ ]:
from google.colab import files
from evidence_fetch.fetch import snapshot_from_html

for s in skipped:
    print(f"\n{s['evidence_id']}  {s['source_url']}\nUpload the saved page (or Cancel to leave it out):")
    try:
        up = files.upload()
    except Exception:
        up = {}
    if not up:
        print("   left out")
        continue
    html = next(iter(up.values())).decode("utf-8", errors="replace")
    snap = snapshot_from_html(html, s["source_url"].strip())
    snaps[s["evidence_id"]] = snap
    print(f"   OK: {len(snap['text'].split())} words, page date: {snap['page_date'] or 'none shown'}")

### Step 6. Check and save to Drive
The cell shows the start of every source. If it looks right, it saves `<company_id>.md` in YL004B evidence.
If the file already exists, it is **replaced** (one file per company, so Make always finds exactly one).

In [ ]:
import io
from googleapiclient.http import MediaIoBaseUpload
from evidence_fetch.fetch import to_source_block

assert snaps, "No source was read. Nothing to save."
order = [s["evidence_id"] for s in sources if s["evidence_id"] in snaps]   # keep the database order
block = "\n".join(to_source_block(snaps[e]) for e in order)

for e in order:
    print(f"--- {e}: {snaps[e]['url']}")
    print("   " + "\n   ".join(snaps[e]["text"].splitlines()[:4]))
print(f"\n{len(order)} sources, {len(block.split())} words. Left out: {[s['evidence_id'] for s in sources if s['evidence_id'] not in snaps] or 'none'}")

FILE_NAME = f"{COMPANY_ID}.md"
media = MediaIoBaseUpload(io.BytesIO(block.encode("utf-8")), mimetype="text/markdown", resumable=False)
found = drive.files().list(
    q=f"'{EVIDENCE_FOLDER_ID}' in parents and name = '{FILE_NAME}' and trashed = false",
    fields="files(id)", supportsAllDrives=True, includeItemsFromAllDrives=True).execute()["files"]

if found:
    f = drive.files().update(fileId=found[0]["id"], media_body=media, fields="id, name, webViewLink",
                             supportsAllDrives=True).execute()
    print(f"\nReplaced: {f['name']}  {f['webViewLink']}")
    if len(found) > 1:
        print(f"WARNING: {len(found)} files with this name in the folder. Make reads only one: remove the extra ones.")
else:
    f = drive.files().create(body={"name": FILE_NAME, "parents": [EVIDENCE_FOLDER_ID]}, media_body=media,
                             fields="id, name, webViewLink", supportsAllDrives=True).execute()
    print(f"\nSaved: {f['name']}  {f['webViewLink']}")
print(f"Next: run the Make Teaser Generator for {LABEL} ({COMPANY_ID}).")